In [ ]:
# Colab Environment Setup: Auto-clone repository assets if running in the cloud
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("[Colab Detected] Cloning repository assets...")
    !git clone https://github.com/mattjunior039/CampusAIAssistantTutorial.git _repo_tmp
    !cp -r _repo_tmp/data .
    !cp _repo_tmp/*.pdf . 2>/dev/null || true
    !rm -rf _repo_tmp
    print("[Colab Ready] Datasets and handbook documents synchronized.")


# Project 1: AI Campus Assistant Pipeline
## Phase 1: Lexical Search & Rule-Based Matching 

---

### Learning Objectives
By completing this hands-on laboratory, you will:
1. **Deconstruct the NLP Evolution:** Contrast discrete lexical retrieval (TF-IDF), dense semantic representation (bi-encoders), and generative orchestration (RAG).
2. **Master the Core Concepts of Vector Space Models:** Understand how Term Frequency-Inverse Document Frequency (TF-IDF) acts as a "popularity penalty," and how Cosine Similarity matches keyword recipes.
3. **Build an End-to-End Production Lexical Search Engine:** Implement text sanitation, tokenization, stop-word elimination, TF-IDF vectorization, inverted indexing, and ranked retrieval.
4. **Empirically Diagnose Lexical Retrieval Failures:** Observe and analyze *Synonym Blindness* (vocabulary mismatch), *Polysemy / Context Blindness* (ambiguous token collisions), and *Sparsity Explosion*.
5. **Implement Robust Fallbacks & Evaluate Performance:** Build automated thresholding, investigate n-gram dimensionality scaling, and execute unit tests.


## 1. Course Introduction & The NLP Evolution Roadmap

Modern conversational AI systems—such as university campus assistants, customer support bots, and enterprise search platforms—did not emerge overnight. They represent a three-decade evolution from discrete symbolic methods to continuous dense vector spaces, and ultimately to generative foundation models.

```
+---------------------------------------------------------------------------------------------------+
|                                 THE NLP & IR EVOLUTIONARY SPECTRUM                                |
+------------------------------------+----------------------------------+---------------------------+
| Phase 1: Lexical & Rule-Based      | Phase 2: Dense Semantic Matching | Phase 3: RAG & GenAI      |
| (1990s - 2010s)                    | (2018 - 2022)                    | (2023 - Present)          |
+------------------------------------+----------------------------------+---------------------------+
| - Bag-of-Words & TF-IDF            | - Word2Vec, GloVe, FastText      | - LLM Generation (GPT-4o) |
| - Inverted Indexes & BM25          | - Dense Bi-Encoders (SBERT)      | - Vector Databases (HNSW) |
| - Exact token & character overlap  | - Continuous embedding spaces    | - Hybrid Retrieval + Rerank|
| - Fast, interpretable, rigid       | - Handles synonyms & paraphrasing| - Context-grounded synthesis|
| * THIS LAB (Phase 1) *             | * NEXT LAB (Phase 2) *           | * FINAL PROJECT (Phase 3) *|
+------------------------------------+----------------------------------+---------------------------+
```

### The Core Premise: Why Lexical Search Fails Human Intent
Lexical search operates under a fundamental assumption: **relevance is a function of shared surface-form tokens**. If a user's query and a reference document share words, they are assumed to be semantically related.

However, natural human language violates this assumption in three critical ways:
1. **Synonymy (The Vocabulary Mismatch Problem):** Different words express identical concepts (e.g., *"car"* vs. *"automobile"*, *"tuition deposit"* vs. *"bursar payment"*). A pure lexical engine produces a similarity score of zero when vocabulary does not overlap.
2. **Polysemy & Ambiguity:** A single token carries distinct meanings depending on context (e.g., *"river bank"* vs. *"commercial bank"*, *"Python course"* vs. *"ball python"*). Lexical systems blindly match the token regardless of meaning.
3. **Word Order & Negation Blindness:** Bag-of-words architectures treat text as unordered multisets. Consequently, *"The exam is not hard, it is easy"* and *"The exam is not easy, it is hard"* produce identical vector representations.


## 2. Step-by-Step Implementation Pipeline

Let us construct our lexical search engine from scratch through a modular, typed architecture.

```
+-------------------------------------------------------------------------------------+
|                              LEXICAL RETRIEVAL PIPELINE                             |
+-------------------------------------------------------------------------------------+
|                                                                                     |
|  [Raw User Query]                          [Campus Knowledge Base: 20 FAQs]        |
|          |                                                |                         |
|          v                                                v                         |
|  +------------------------+                     +------------------------+          |
|  | Regex Sanitization     |                     | Regex Sanitization     |          |
|  | Lowercasing & Punct.   |                     | Lowercasing & Punct.   |          |
|  | NLTK Tokenization      |                     | NLTK Tokenization      |          |
|  | Stop-Word Filtering    |                     | Stop-Word Filtering    |          |
|  +------------------------+                     +------------------------+          |
|          |                                                |                         |
|          v                                                v                         |
|  [Sanitized Tokens]                             [Sanitized Corpus Matrix]           |
|          |                                                |                         |
|          v                                                v                         |
|  +--------------------------------------------------------------------+             |
|  |                  scikit-learn TfidfVectorizer                      |             |
|  |            Vocabulary V, Document-Term Matrix (N x |V|)            |             |
|  +--------------------------------------------------------------------+             |
|                                |                                                    |
|                                v                                                    |
|            +---------------------------------------+                                |
|            | Cosine Similarity (Query dot Docs)    |                                |
|            | Matched Feature Contribution Extraction|                                |
|            | Top-K Ranking & Threshold Filtering   |                                |
|            +---------------------------------------+                                |
|                                |                                                    |
|                                v                                                    |
|                  [Ranked Results + Diagnostics]                                     |
+-------------------------------------------------------------------------------------+
```


In [ ]:
# Step 3.1: Environment Setup & Verification
import sys
import subprocess
from typing import List, Dict, Any, Tuple, Optional

# Install required packages if missing in the environment
required_packages = ["scikit-learn", "nltk", "pandas", "numpy", "matplotlib", "seaborn", "ipywidgets"]
for pkg in required_packages:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"[Setup] Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import ipywidgets as widgets
from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Download required NLTK tokenizers and stop-word corpora
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

print(f"[OK] Python version: {sys.version.split()[0]}")
print(f"[OK] NumPy version: {np.__version__}")
print(f"[OK] Pandas version: {pd.__version__}")

print(f"[OK] NLTK & Scikit-Learn successfully configured.")
print(f"[OK] ipywidgets ready for interactive TF-IDF exploration.")

### Campus FAQ Knowledge Base Synthesis

To make our pedagogical failure analysis transparent, we synthesize an in-memory dataset of 20 realistic university FAQ entries.

In [ ]:
# Step 3.2: Define the 20 Campus FAQ Knowledge Base Entries

FAQ_DATA: List[Dict[str, Any]] = [
    {
        "faq_id": "FAQ-01",
        "category": "Parking & Transportation",
        "question": "How do students register for an on-campus automobile storage permit?",
        "answer": "Automobile storage permits must be requested via the Department of Motor Transportation portal with vehicle registration documentation."
    },
    {
        "faq_id": "FAQ-02",
        "category": "Financial Services",
        "question": "What is the mandatory procedure for bursar fee deposit and wire remittance?",
        "answer": "All bursar fee deposit and tuition wire remittance procedures must clear through the central treasury portal prior to the third academic calendar week."
    },
    {
        "faq_id": "FAQ-03",
        "category": "Residential Life",
        "question": "What regulations govern residential living quad dwelling room assignments?",
        "answer": "Undergraduate residential living quad dwelling allocations are finalized every July by the Office of Student Housing Administration."
    },
    {
        "faq_id": "FAQ-04",
        "category": "Academic Support",
        "question": "Where can undergraduates receive pedagogical consultation and academic remediation?",
        "answer": "Pedagogical consultation and peer remediation workshops take place Mondays through Thursdays at the Academic Success Pavilion."
    },
    {
        "faq_id": "FAQ-05",
        "category": "Library & Research",
        "question": "How many books and research manuscripts can a student borrow from the central library?",
        "answer": "Undergraduate scholars may borrow up to twenty-five print monographs and research manuscripts for a four-week renewable loan period."
    },
    {
        "faq_id": "FAQ-06",
        "category": "Health & Wellness",
        "question": "Where is the student health and wellness clinical infirmary located?",
        "answer": "The student health center and clinical infirmary is located on North Campus adjacent to the recreation facility."
    },
    {
        "faq_id": "FAQ-07",
        "category": "Dining Services",
        "question": "How do meal plans and dining hall culinary swipe credits work?",
        "answer": "Campus culinary swipe credits reload automatically on Sunday midnight and can be utilized across all three residential dining commons."
    },
    {
        "faq_id": "FAQ-08",
        "category": "Information Technology",
        "question": "How do I reset my campus network credentials and institutional password?",
        "answer": "Visit the campus identity management portal and complete multi-factor authentication to initiate an institutional password reset."
    },
    {
        "faq_id": "FAQ-09",
        "category": "Recreation & Athletics",
        "question": "What are the operating hours for the campus aquatic center and gymnasium?",
        "answer": "The varsity gymnasium and aquatic swimming complex are open daily from 6:00 AM until 11:00 PM with valid student ID access."
    },
    {
        "faq_id": "FAQ-10",
        "category": "Career Development",
        "question": "How do I schedule an appointment with a career development counselor?",
        "answer": "Log into the university career network portal to book mock interviews, resume critiques, and internship advising sessions."
    },
    {
        "faq_id": "FAQ-11",
        "category": "Registrar & Enrollment",
        "question": "What is the official deadline to drop a course without academic transcript penalty?",
        "answer": "The deadline to drop an academic course without receiving a withdrawal mark on your transcript is the end of the tenth instructional day."
    },
    {
        "faq_id": "FAQ-12",
        "category": "Financial Services",
        "question": "Which commercial bank institution handles university wire transfers and student deposits?",
        "answer": "The university partners with First State Bank for institutional escrow, international tuition wire transfers, and student deposit accounts."
    },
    {
        "faq_id": "FAQ-13",
        "category": "Campus Safety",
        "question": "How do students summon university police emergency escort services after hours?",
        "answer": "Dial 555-SAFE from any blue-light callbox station on campus to request an immediate university police escort to your residence."
    },
    {
        "faq_id": "FAQ-14",
        "category": "Accessibility Services",
        "question": "How do I register for disability classroom accommodations and exam proctoring?",
        "answer": "Submit medical documentation to the Accessibility Resources Office at least four weeks prior to midsemester examination periods."
    },
    {
        "faq_id": "FAQ-15",
        "category": "Student Activities",
        "question": "How can students charter a new registered campus student organization or club?",
        "answer": "New student organizations require at least ten enrolled active members, a faculty advisor, and approval from the Student Union Senate."
    },
    {
        "faq_id": "FAQ-16",
        "category": "International Student Services",
        "question": "How do international scholars maintain F-1 visa compliance and employment authorization?",
        "answer": "F-1 visa holders must maintain full-time academic enrollment (minimum twelve credits) and secure designated school official authorization before off-campus employment."
    },
    {
        "faq_id": "FAQ-17",
        "category": "Facilities & Maintenance",
        "question": "How do I submit an urgent maintenance repair work order for my dorm room?",
        "answer": "Submit a facilities work order request through the campus housing portal for plumbing, electrical, heating, or key replacement repairs."
    },
    {
        "faq_id": "FAQ-18",
        "category": "Graduation & Commencement",
        "question": "What are the graduation requirements and commencement regalia ordering procedures?",
        "answer": "Students must complete all departmental capstone requirements and order their graduation cap and gown regalia via the university bookstore by April 1."
    },
    {
        "faq_id": "FAQ-19",
        "category": "Sustainability & Waste",
        "question": "Where are zero-waste compost receptacles and electronic recycling bins located?",
        "answer": "Compost receptacles and electronic waste recycling drop-off stations are stationed in the lobby of every academic building and dining commons."
    },
    {
        "faq_id": "FAQ-20",
        "category": "Outdoor Recreation",
        "question": "Are students permitted to fish or kayak along the scenic campus river bank?",
        "answer": "Recreational non-motorized kayaking is allowed on the river, but fishing along the campus river bank requires a state conservation permit."
    }
]

# Convert into a structured pandas DataFrame
faq_df = pd.DataFrame(FAQ_DATA)

# Combine Question + Answer to form rich, retrievable document text
faq_df["full_text"] = faq_df["question"] + " " + faq_df["answer"]

print(f"[Corpus] Successfully initialized {len(faq_df)} FAQ knowledge base documents.")
print(f"[Corpus] Dataset Columns: {list(faq_df.columns)}")
display(faq_df[["faq_id", "category", "question"]].head(5))


### Text Normalization & Preprocessing Pipeline

Before numerical vectorization, raw natural text must be transformed into clean, canonical tokens through a reproducible pipeline:

1. **Lowercasing** 
2. **Regex Punctuation Removal** 
3. **NLTK Word Tokenization** 
4. **Custom Stop-Word Engineering**
   - Standard NLTK stop-words are useful, but they can also remove important domain context.
   - For example, a word like **"fee"** or **"permit"** might be highly relevant to campus support, even though it is not a grammatical filler word.
   - Students can experiment by appending or removing words from the stop-word list to see how retrieval quality changes.

This is your chance to design a preprocessing strategy instead of copying one blindly.

---

### 3.3A Hands-On Challenge: Break the Engine on Purpose
Try to craft a user query that is semantically very close to a campus FAQ, but still produces a cosine similarity of `0.0`.

Your goal is to intentionally break the lexical system by using a different vocabulary, even though the meaning is almost identical.

Examples of the type of challenge you should try:
- **"Where can I park my car?"** vs. an FAQ entry using **"automobile storage permit"**
- **"How do I pay tuition?"** vs. an FAQ entry using **"bursar fee deposit and wire remittance"**
- **"I need dorm repair help"** vs. an FAQ entry using **"residential living quad dwelling allocations"**

If you can design a prompt that the system misses, you have discovered the exact weakness that motivates Phase 2.


In [ ]:
# =============================================================================
# Step 3.3: Preprocessing Pipeline Implementation
# =============================================================================

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re

# 1. CUSTOM STOP-WORD ENGINEERING
# The default NLTK English stop-words (like "the", "is", "at")
ENGLISH_STOPWORDS: set = set(stopwords.words("english"))

# STUDENT CHALLENGE: Add or remove domain-specific words after inspecting the preview.
# ENGLISH_STOPWORDS.add("university")
# ENGLISH_STOPWORDS.remove("few")

# -----------------------------------------------------------------------------
# 2. LIVE PREPROCESSING PREVIEW
# See each transformation before completing tokenize_and_clean below.
# -----------------------------------------------------------------------------
def preview_tokenization(text: str) -> None:
    text_lower = text.lower()
    text_clean = re.sub(r"[^a-zA-Z0-9\s]", " ", text_lower)
    raw_tokens = word_tokenize(text_clean)
    content_tokens = [
        token for token in raw_tokens
        if token not in ENGLISH_STOPWORDS and len(token) >= 1
    ]

    print(f"Lowercased:       {text_lower}")
    print(f"Tokenized:        {raw_tokens}")
    print(f"After stop words: {content_tokens}")


preview_text = widgets.Text(
    value="Tuition deposits for F-1 visa students are due by Friday!",
    placeholder="Type a sentence...",
    description="Sentence:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="700px"),
)
preview_output = widgets.Output()


def update_token_preview(change: dict | None = None) -> None:
    with preview_output:
        preview_output.clear_output(wait=True)
        preview_tokenization(preview_text.value)


preview_text.observe(update_token_preview, names="value")
display(preview_text, preview_output)
update_token_preview()

# -----------------------------------------------------------------------------
# 3. THE PIPELINE FUNCTION: COMPLETE THE TODOs
# -----------------------------------------------------------------------------
def tokenize_and_clean(text: str, remove_stopwords: bool = True) -> list[str]:
    """Cleans a raw string into a list of normalized token strings."""
    if not isinstance(text, str):
        return []

    # 1. LOWERCASE: Convert 'text' to lowercase.
    text_lower = ...  # TODO: text.lower()

    # 2. PUNCTUATION: Replace anything that is not a letter or number with a space.
    text_clean = ...  # TODO: re.sub(r"[^a-zA-Z0-9\s]", " ", text_lower)

    # 3. TOKENIZE: Break the clean string into words with NLTK.
    tokens = ...  # TODO: word_tokenize(text_clean)

    # 4. FILTER: Keep non-empty tokens and optionally remove stop words.
    final_tokens = []
    # TODO: Write the filtering loop using the live preview as your guide.

    return final_tokens

# -----------------------------------------------------------------------------
# 4. VERIFICATION
# -----------------------------------------------------------------------------
sample_raw_sentences = [
    "How do international scholars maintain F-1 visa compliance?",
    "Where is the student health & wellness clinical infirmary located?",
    "Are students permitted to fish along the scenic campus river bank?",
]

print(f"{'RAW TEXT':<70} | {'SANITIZED TOKENS'}")
print("-" * 115)
for sentence in sample_raw_sentences:
    cleaned = tokenize_and_clean(sentence)
    print(f"{sentence:<70} | {cleaned}")

In [ ]:
# Self-Check Unit Test: Preprocessing Pipeline
def test_preprocessing():
    test_str = "What is the fee for 2026 bursar wire deposits for F-1 visa students??!"
    tokens = tokenize_and_clean(test_str, remove_stopwords=True)
    
    assert isinstance(tokens, list), "Output must be a list of tokens."
    assert "what" not in tokens, "Stop words like 'what' must be filtered."
    assert "is" not in tokens, "Stop words like 'is' must be filtered."
    assert "fee" in tokens, "'fee' should be preserved as an informative token."
    assert "2026" in tokens, "Alphanumeric tokens should be preserved."
    assert "deposits" in tokens, "'deposits' should be preserved."
    assert "f" in tokens and "1" in tokens, "Single alphanumeric characters like 'f' and '1' from F-1 visa must be preserved."
    assert all(not re.search(r"[^\w\s]", tok) for tok in tokens), "No punctuation allowed in tokens."
    print("[PASS] Preprocessing Pipeline Unit Tests Passed Successfully!")

test_preprocessing()


### 3.4 Vector Space Modeling: Fitting `TfidfVectorizer`

We now fit scikit-learn's `TfidfVectorizer` to our 20 FAQ documents.

We will inspect the resulting document-term matrix and see how the vocabulary changes as the corpus grows.

**The "Make Money" Challenge:**
Type `"How do I make money?"` into the box below. Watch what happens to the chart. Because our NLTK filter strips the grammar (how, do, i), and the words "make" and "money" don't exist in our formal campus dataset, the TF-IDF vectorizer goes completely blind!

In [ ]:
# Step 3.4: Fitting the Harmonized Vector Space Model

# Instantiate the vectorizer with custom tokenizer and token_pattern=None
vectorizer = TfidfVectorizer(
    tokenizer=tokenize_and_clean,
    token_pattern=None,
    ngram_range=(1, 1),
    norm="l2",
    smooth_idf=True
)

# Fit on the combined corpus text (N documents)
doc_matrix = vectorizer.fit_transform(faq_df["full_text"])

# Extract vocabulary and dimensions
vocab = vectorizer.vocabulary_
feature_names = np.array(vectorizer.get_feature_names_out())
num_docs, vocab_size = doc_matrix.shape

# Compute Document-Term Matrix Sparsity
non_zero_elements = doc_matrix.nnz
total_elements = num_docs * vocab_size
sparsity = (1.0 - (non_zero_elements / total_elements)) * 100.0

print(f"================ TF-IDF VECTOR SPACE SUMMARY ================")
print(f"Total Documents in Corpus (N)      : {num_docs}")
print(f"Vocabulary Dimension (|V|)          : {vocab_size} unique terms")
print(f"Sparse Matrix Non-Zero Entries     : {non_zero_elements} / {total_elements}")
print(f"Matrix Sparsity Percentage         : {sparsity:.2f}% sparse")
print(f"=============================================================")

# Display a sample of the learned vocabulary indices
sample_vocab_items = list(vocab.items())[:10]
print(f"Sample Vocabulary Index Mapping: {sample_vocab_items}")


In [ ]:
# Step 3.4 Visualization: Interactive TF-IDF Weight Explorer
# This version makes the idea of "importance weighting" feel tangible.

# Build a helpful interactive explorer for a student's custom query
text_input = widgets.Text(
    value="How do I pay tuition and wire my fee deposit?",
    placeholder="Type a campus question...",
    description="Query:",
    layout=widgets.Layout(width="500px")
)


def plot_query_tfidf(query_text: str):
    if not query_text.strip():
        return

    query_vec = vectorizer.transform([query_text])
    query_weights = query_vec.toarray().flatten()
    term_names = np.array(vectorizer.get_feature_names_out())

    # Keep only terms with non-zero TF-IDF weight in the query
    active = np.where(query_weights > 0)[0]
    if len(active) == 0:
        print("[Info] No weighted words were found in this query.")
        return

    labels = term_names[active]
    values = query_weights[active]

    sorted_idx = np.argsort(values)[::-1]
    labels = labels[sorted_idx]
    values = values[sorted_idx]

    plt.figure(figsize=(10, 5))
    plt.bar(labels[:12], values[:12], color="steelblue")
    plt.title("TF-IDF Weight of Query Terms", fontsize=14, fontweight="bold")
    plt.ylabel("Weight")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    print("\nMost informative query terms:")
    for label, value in zip(labels[:8], values[:8]):
        print(f"  - {label}: {value:.4f}")


widgets.interactive(
    plot_query_tfidf,
    query_text=text_input
)


In [ ]:
# Self-Check Unit Test: Document-Term Matrix Properties
def test_vector_space_properties():
    # 1. Verify dimensions
    assert doc_matrix.shape[0] == len(faq_df), "Document count must match FAQ rows (20)."
    assert doc_matrix.shape[1] == len(vectorizer.vocabulary_), "Matrix columns must equal vocabulary size."
    
    # 2. Verify L2 unit-norm property for non-empty documents
    dense_matrix = doc_matrix.toarray()
    row_norms = np.linalg.norm(dense_matrix, axis=1)
    np.testing.assert_allclose(
        row_norms, 
        np.ones(len(faq_df)), 
        rtol=1e-5, 
        err_msg="Every document vector must have Euclidean L2 norm of 1.0."
    )
    
    # 3. Verify non-negativity
    assert (dense_matrix >= 0.0).all(), "TF-IDF weights must be non-negative."
    print("[PASS] Vector Space Model Properties Validated (Shapes, L2 Unit Norms, Non-negativity)!")

test_vector_space_properties()


### 3.5 Inference & Search Engine Implementation

When a user submits a query:
1. The text is cleaned and transformed into the same vocabulary used by the FAQ corpus.
2. Every document in the corpus gets compared against the query using cosine similarity.
3. The documents are ranked from strongest match to weakest match.
4. The matched terms are surfaced so students can inspect which words caused the result.
5. A confidence threshold can be used to decline weak or irrelevant matches.

The core idea is simple: a document is a good match when its important keywords overlap with the query's important keywords, even if the wording is not identical.


In [ ]:
# Step 3.5: Implement the Lexical Search Engine with Feature Attribution

def search_campus_faq(
    query: str,
    vectorizer: TfidfVectorizer,
    doc_matrix: Any,
    faq_df: pd.DataFrame,
    top_k: int = 3,
    threshold: float = 0.10
) -> List[Dict[str, Any]]:
    """Search the campus FAQ corpus using TF-IDF and cosine similarity."""
    # The vectorization, matrix math, ranking, and array slicing are pre-written.
    query_vec = vectorizer.transform([query])
    similarities = cosine_similarity(query_vec, doc_matrix).flatten()
    ranked_indices = np.argsort(similarities)[::-1][:top_k]
    feature_names = np.array(vectorizer.get_feature_names_out())
    query_dense = query_vec.toarray().flatten()

    valid_matches = []

    for rank, doc_idx in enumerate(ranked_indices, start=1):
        score = float(similarities[doc_idx])
        doc_dense = doc_matrix[doc_idx].toarray().flatten()
        overlap_weights = query_dense * doc_dense
        matched_feature_indices = np.where(overlap_weights > 0)[0]
        matched_keywords = feature_names[matched_feature_indices].tolist()
        row = faq_df.iloc[doc_idx]

        candidate = {
            "rank": rank,
            "faq_id": row["faq_id"],
            "category": row["category"],
            "question": row["question"],
            "answer": row["answer"],
            "cosine_score": score,
            "is_above_threshold": True,
            "matched_keywords": matched_keywords,
        }

        # STUDENT TODO: Uncomment these lines to keep confident matches.
        # if score >= threshold:
        #     valid_matches.append(candidate)

    return valid_matches


def print_search_results(query: str, results: List[Dict[str, Any]]) -> None:
    """Display ranked search results in a readable format."""
    print("\n" + "=" * 80)
    print(f"SEARCH QUERY: \"{query}\"")
    print("=" * 80)
    if not results:
        print("No results met the confidence threshold.")
        return

    for result in results:
        print(f"Rank {result['rank']} | Score: {result['cosine_score']:.4f} [VALID MATCH]")
        print(f"FAQ ID: {result['faq_id']} | Category: {result['category']}")
        print(f"Question: {result['question']}")
        print(f"Answer:   {result['answer']}")
        print(f"Matched Tokens: {result['matched_keywords']}")
        print("-" * 80)

In [ ]:
# Self-Check Unit Test: Search Engine Retrieval Output
def test_search_engine():
    results = search_campus_faq(
        query="bursar fee deposit wire",
        vectorizer=vectorizer,
        doc_matrix=doc_matrix,
        faq_df=faq_df,
        top_k=3,
        threshold=0.1,
    )

    assert 1 <= len(results) <= 3, "Return only confident matches, capped at top_k."
    assert all(
        first["cosine_score"] >= second["cosine_score"]
        for first, second in zip(results, results[1:])
    ), "Results must be sorted descending."
    assert all(result["cosine_score"] >= 0.1 for result in results), "Every result must meet the threshold."
    assert 0.0 <= results[0]["cosine_score"] <= 1.0, "Cosine score must be bounded within [0, 1]."
    assert "faq_id" in results[0] and "matched_keywords" in results[0], "Expected output keys missing."
    print("[PASS] Search Engine Basic Invariants Validated Successfully!")


test_search_engine()

## 4. Student Lab Exercises & Deliberate Failure Analysis

Now comes the core pedagogical inquiry of Phase 1: **Where and why does Lexical Search break down?**

Use the dropdown below to switch between four controlled scenarios and watch the search engine's results and diagnosis update live:

1. **The Happy Path (Exact Token Overlap):** When the user speaks the exact vocabulary of the knowledge base.
2. **Synonym Blindness (Lexical Fragility):** When the user asks a completely natural question, but uses synonyms absent from the index.
3. **Polysemy & Context Blindness (False-Positive Retrieval):** When identical tokens represent completely distinct concepts in different domains.


In [ ]:
# =============================================================================
# INTERACTIVE FAILURE ANALYSIS: EXPLORE HOW LEXICAL SEARCH BREAKS
# =============================================================================

SCENARIO_EXPLANATIONS = {
    "bursar fee deposit and wire remittance": (
        "[HAPPY PATH]: Perfect lexical overlap with FAQ-02's exact vocabulary "
        "('bursar', 'fee', 'deposit', 'wire', 'remittance'). Cosine similarity is strong "
        "because both term frequency and inverse document frequency align precisely."
    ),
    "Where can I park my car?": (
        "[SYNONYM BLINDNESS]: A human understands 'park my car' means the same as "
        "'automobile storage', but the TF-IDF dot product is near 0.0 because the query "
        "tokens {'park', 'car'} and document tokens {'automobile', 'storage', 'permit'} "
        "share no vocabulary. This is the Vocabulary Mismatch Problem that motivates Phase 2."
    ),
    "Can I fish or kayak on the campus river bank?": (
        "[POLYSEMY - RIVER BANK]: The token 'bank' is shared with FAQ-12 (Commercial Bank). "
        "Bag-of-Words treats 'bank' as one feature regardless of meaning, so it can partially "
        "activate the wrong FAQ even though 'river bank' and 'investment bank' are unrelated."
    ),
    "Which commercial bank handles tuition wire transfers?": (
        "[POLYSEMY - FINANCIAL BANK]: This query should correctly match FAQ-12, but notice how "
        "the shared token 'bank' gives lexical search no way to distinguish this from the "
        "river-bank scenario above. Bag-of-words has no notion of word sense."
    ),
}


def explore_failure_cases(query: str) -> None:
    results = search_campus_faq(query, vectorizer, doc_matrix, faq_df, top_k=2)
    print_search_results(query, results)
    print(SCENARIO_EXPLANATIONS.get(query, "No diagnostic note available for this scenario."))


widgets.interact(
    explore_failure_cases,
    query=widgets.Dropdown(
        options=[
            ("Happy Path (Exact Overlap)", "bursar fee deposit and wire remittance"),
            ("Synonym Blindness (Vocabulary Mismatch)", "Where can I park my car?"),
            ("Polysemy (River Bank)", "Can I fish or kayak on the campus river bank?"),
            ("Polysemy (Financial Bank)", "Which commercial bank handles tuition wire transfers?"),
        ],
        description="Scenario:",
        layout=widgets.Layout(width="550px"),
    ),
);


## 5. Interactive N-Gram Dimensionality Lab

### N-Gram Range Exploration & Dimensional Explosion

Our original vectorizer uses unigrams with `ngram_range=(1, 1)`. Expanding the range to include bigrams `(1, 2)` or trigrams `(1, 3)` preserves short phrases such as `river bank` and `commercial bank`, but creates many more vocabulary dimensions.

Use the range slider below to move from individual words toward multi-word phrases. The chart compares your selection with the unigram baseline:

- **Vocabulary size $|V|$** grows as more phrases become distinct features.
- **Matrix density** falls as those extra feature columns remain empty for most documents. Equivalently, matrix sparsity rises.

#### What is an N-Gram?

- **Unigram:** one word, such as `commercial`
- **Bigram:** two adjacent words, such as `commercial bank`
- **Trigram:** three adjacent words, such as `the commercial bank`

Move the handles from `(1, 1)` through `(1, 3)` and watch the vector space change immediately.

In [ ]:
# Interactive N-Gram Dimensionality Explorer

ngram_slider = widgets.IntRangeSlider(
    value=(1, 1),
    min=1,
    max=3,
    step=1,
    description="N-gram range:",
    continuous_update=True,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px"),
)


def measure_ngram_space(ngram_range: tuple[int, int]) -> dict:
    """Fit one n-gram configuration and return its dimensionality metrics."""
    experiment_vectorizer = TfidfVectorizer(
        tokenizer=tokenize_and_clean,
        token_pattern=None,
        ngram_range=ngram_range,
        norm="l2",
    )
    experiment_matrix = experiment_vectorizer.fit_transform(faq_df["full_text"])
    document_count, vocabulary_size = experiment_matrix.shape
    total_slots = document_count * vocabulary_size
    density = (experiment_matrix.nnz / total_slots) * 100.0
    sparsity = 100.0 - density
    return {
        "range": ngram_range,
        "vocabulary_size": vocabulary_size,
        "density": density,
        "sparsity": sparsity,
    }


unigram_metrics = measure_ngram_space((1, 1))


def update_ngram_chart(ngram_range: tuple[int, int]) -> None:
    selected_metrics = measure_ngram_space(ngram_range)
    labels = ["Unigrams\n(1, 1)", f"Selected\n{ngram_range}"]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    vocabulary_bars = axes[0].bar(
        labels,
        [unigram_metrics["vocabulary_size"], selected_metrics["vocabulary_size"]],
        color=["#4C78A8", "#E45756"],
    )
    axes[0].bar_label(vocabulary_bars, padding=3)
    axes[0].set_title("Vocabulary Size $|V|$")
    axes[0].set_ylabel("Unique features")

    density_bars = axes[1].bar(
        labels,
        [unigram_metrics["density"], selected_metrics["density"]],
        color=["#59A14F", "#F28E2B"],
    )
    axes[1].bar_label(density_bars, fmt="%.2f%%", padding=3)
    axes[1].set_title("Document-Term Matrix Density")
    axes[1].set_ylabel("Non-zero cells (%)")
    axes[1].set_ylim(0, max(unigram_metrics["density"], selected_metrics["density"]) * 1.25)

    fig.suptitle(
        f"Selected {ngram_range}: {selected_metrics['sparsity']:.2f}% sparse",
        fontsize=13,
    )
    fig.tight_layout()
    plt.show()


widgets.interact(update_ngram_chart, ngram_range=ngram_slider);

---

### Task B: Production Fallback Mechanism with Confidence Thresholding
In a live campus assistant, returning an irrelevant FAQ with a low similarity score (e.g., 0.04) causes user frustration and hallucinations.

**Your Objective:** Implement an enhanced retrieval function `search_with_fallback(...)` that:
1. Evaluates if the top retrieved result meets a confidence threshold $	au$ (e.g., $	au = 0.20$).
2. If `top_score < threshold`, returns a structured fallback payload directing the student to campus human support, rather than delivering incorrect information.

What happens if a student asks the Campus Assistant: *"Where can I buy vegan gluten-free pizza near the football stadium?"*

Our current search engine will look at those words, find a weak match for "stadium" in the parking FAQ, calculate a terrible similarity score of 0.04, and **still return the parking FAQ to the user.** That creates a frustrating user experience.

To fix this, we build a **Confidence Threshold**. Think of it as a bouncer at the door of your search engine. The bouncer checks the cosine score of the very best result. If that score is under our required limit (like $0.20$), the bouncer rejects the result and hands the user a helpful "I don't know" message instead.

Here is the scaffolded code to build the bouncer:

In [ ]:
# =============================================================================
# STUDENT TASK B: THE CONFIDENCE BOUNCER (FALLBACK MECHANISM)
# =============================================================================

def search_with_fallback(
    query: str,
    vectorizer: TfidfVectorizer,
    doc_matrix: Any,
    faq_df: pd.DataFrame,
    top_k: int = 3,
    confidence_threshold: float = 0.20
) -> dict:
    """
    Acts as a safety guard. If the search engine is guessing, it intercepts the bad 
    answer and returns a helpful error payload instead.
    """
    # 1. RUN THE SEARCH
    # We call the function you built in Step 3.5 to get the raw results
    raw_results = search_campus_faq(
        query=query,
        vectorizer=vectorizer,
        doc_matrix=doc_matrix,
        faq_df=faq_df,
        top_k=top_k,
        threshold=confidence_threshold
    )
    
    # 2. FIND THE BEST SCORE
    # Grab the #1 result. If the results list is empty, default to a score of 0.0
    top_result = raw_results[0] if raw_results else None
    top_score = top_result["cosine_score"] if top_result else 0.0
    
    # 3. THE BOUNCER LOGIC
    # Hint: Check if the top_score is LESS THAN the confidence_threshold
    if ...:  # TODO: Replace '...' with the correct math comparison
        
        # 4. BUILD THE FALLBACK PAYLOAD
        # Return a dictionary so the website/app knows to show an error message
        return {
            "status": "...",  # TODO: Replace '...' with "FALLBACK_TRIGGERED"
            "query": query,
            "max_confidence_score": top_score,
            "fallback_message": (
                "I'm sorry, I could not find a verified campus policy matching your exact wording. "
                "Please contact the Student Services Central Desk at help@campus.edu."
            ),
            "suggested_actions": [
                "Try rephrasing your question using official administrative terms.",
                "Check the campus directory at directory.campus.edu."
            ]
        }
    else:
        # 5. BUILD THE SUCCESS PAYLOAD
        return {
            "status": "...",  # TODO: Replace '...' with "SUCCESS"
            "query": query,
            "max_confidence_score": top_score,
            "results": raw_results
        }

# -----------------------------------------------------------------------------
# TEST THE BOUNCER
# -----------------------------------------------------------------------------
# We will ask an Out-Of-Vocabulary (OOV) question that has nothing to do with campus policies.
crazy_query = "Where can I buy vegan gluten-free pizza near the football stadium?"

fallback_response = search_with_fallback(
    query=crazy_query, 
    vectorizer=vectorizer, 
    doc_matrix=doc_matrix, 
    faq_df=faq_df, 
    confidence_threshold=0.20
)

print(f"Query: '{crazy_query}'")
print(f"Status: {fallback_response['status']}")
print(f"Max Cosine Score: {fallback_response['max_confidence_score']:.4f}")
print(f"System Message: {fallback_response.get('fallback_message')}")

In [ ]:
# =============================================================================
# SECTION 6: COMPREHENSIVE STUDENT SELF-CHECK TEST SUITE
# =============================================================================

def run_comprehensive_self_check():
    print("[Testing Suite] Initiating comprehensive verification checks...")
    
    # Test 1: Preprocessor idempotence and stop words
    s1 = "The university is closed for winter break!"
    toks1 = tokenize_and_clean(s1)
    assert "university" in toks1, "Content token 'university' must be retained."
    assert "the" not in toks1 and "is" not in toks1 and "for" not in toks1, "Stopwords must be stripped."
    print("  ✓ Check 1: Text sanitization and stop-word filtering verified.")
    
    # Test 2: Vocabulary bounds
    assert len(vectorizer.vocabulary_) > 100, "Vocabulary must contain over 100 domain tokens."
    print("  ✓ Check 2: Vocabulary dimension verified.")
    
    # Test 3: Cosine Similarity mathematical boundary conditions
    # Identical text must yield cosine similarity == 1.0 (within numerical float tolerance)
    exact_q = faq_df.loc[0, "full_text"]
    q_vec = vectorizer.transform([exact_q])
    sim_exact = cosine_similarity(q_vec, doc_matrix[0:1])[0][0]
    np.testing.assert_allclose(sim_exact, 1.0, rtol=1e-4, err_msg="Self-similarity of identical text must equal 1.0")
    print("  ✓ Check 3: Identity vector cosine projection (sim == 1.0) verified.")
    
    # Test 4: Orthogonal / Zero-overlap queries must yield similarity == 0.0
    gibberish_q = "xyzqwk123 nonexistingtoken999"
    res_gibberish = search_campus_faq(gibberish_q, vectorizer, doc_matrix, faq_df)
    assert res_gibberish[0]["cosine_score"] == 0.0, "Disjoint vocabulary query must yield exact 0.0 cosine similarity."
    print("  ✓ Check 4: Orthogonal disjoint vocabulary handling (sim == 0.0) verified.")
    
    # Test 5: Fallback trigger thresholding
    fb_res = search_with_fallback(gibberish_q, vectorizer, doc_matrix, faq_df, confidence_threshold=0.25)
    assert fb_res["status"] == "FALLBACK_TRIGGERED", "Fallback must trigger on zero-score queries."
    print("  ✓ Check 5: Automated fallback threshold logic verified.")
    
    print("\n" + "=" * 80)
    print("🎉 ALL SELF-CHECK UNIT TESTS PASSED WITH ZERO ERRORS!")
    print("=" * 80)

run_comprehensive_self_check()


## 6. Summary, Critical Reflection & Transition to Phase 2

### Summary of Phase 1 Key Findings

| Metric / Dimension | Lexical Search (TF-IDF + Cosine Similarity) |
| :--- | :--- |
| **Computational Complexity** | Highly efficient: sparse dot-product inference in $\mathcal{O}(|V_{\text{active}}|)$ time. |
| **Training Requirements** | Unsupervised; no GPU or neural gradient updates required. |
| **Explainability** | $100\%$ interpretable via individual term TF-IDF dot-product attribution. |
| **Synonym Handling** | **Fails completely ($0.0$ similarity)** without manual synonym thesauri. |
| **Polysemy Handling** | **Blind to context** (matches ambiguous words like *"bank"* across unrelated domains). |
| **Word Order / Negation** | Ignored by Bag-of-Words assumption. |

---

### Student Reflection Questions (Pre-Lab for Phase 2)
1. If we added stemming (e.g., Porter Stemmer) or lemmatization (WordNet Lemmatizer) to our pipeline, which failure modes would be mitigated, and which would remain unsolved?
2. Why can't we simply build an exhaustive dictionary of all synonyms for every word on a university campus? (Consider domain drift, slang, polysemy, and maintenance overhead).
3. How do continuous dense embedding spaces (e.g., Sentence Transformers / SBERT) represent the sentence *"Where can I park my car?"* such that its cosine similarity to *"automobile storage permits"* is $>0.85$, despite sharing **zero** vocabulary tokens?

---

**Next Up — Phase 2:** *Dense Semantic Representations, Sentence-BERT Bi-Encoders & Approximate Nearest Neighbors (ANN).*
